# MedSpan — Biomedical Extractive QA with BioBERT

**Model:** `dmis-lab/biobert-base-cased-v1.2` fine-tuned for extractive QA
via HuggingFace `AutoModelForQuestionAnswering`.

**Dataset:** BioASQ Task B factoid subset loaded via `bigbio/bioasq` on HuggingFace.

**Runtime targets:**
- Google Colab T4 — full dataset, CUDA
- Apple M1 Mac — 100-example debug subset, MPS


## 0. Install dependencies

In [ ]:
import subprocess, sys, pathlib

PKGS = [
    "torch==2.2.2",
    "transformers==4.40.1",
    "datasets==2.19.0",
    "tokenizers==0.19.1",
    "accelerate==0.29.3",
    "evaluate==0.4.1",
    "gradio==4.28.3",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "scikit-learn==1.4.2",
    "tqdm==4.66.2",
]

if pathlib.Path("requirements.txt").exists():
    # Prefer the pinned file when running from the cloned repo
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
else:
    # Fallback: install packages listed above directly (e.g. fresh Colab session)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + PKGS,
        check=True,
    )

print("Dependencies ready.")

## 1. Imports & device detection

In [ ]:
import os
import json
import collections
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Device selection
# ---------------------------------------------------------------------------
# CUDA  → full dataset on Colab T4
# MPS   → debug subset on Apple M1/M2 (Metal Performance Shaders)
# CPU   → fallback for any other machine

if torch.cuda.is_available():
    DEVICE      = "cuda"
    DEBUG_MODE  = False          # use full dataset
    DEBUG_N     = None
    print(f"CUDA detected: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE      = "mps"
    DEBUG_MODE  = True           # keep only 100 examples to stay fast
    DEBUG_N     = 100
    print("Apple MPS detected — running in debug mode (100 examples).")
else:
    DEVICE      = "cpu"
    DEBUG_MODE  = True
    DEBUG_N     = 100
    print("No GPU detected — running in debug mode on CPU.")

print(f"Active device : {DEVICE}")
print(f"Debug mode    : {DEBUG_MODE}")

## 2. Configuration

In [ ]:
# ---------------------------------------------------------------------------
# All tuneable hyper-parameters live here so nothing is buried in the code.
# ---------------------------------------------------------------------------

MODEL_NAME   = "dmis-lab/biobert-base-cased-v1.2"
SAVE_DIR     = "./saved_model"

MAX_LENGTH   = 384   # max total token length (question + context)
DOC_STRIDE   = 128   # overlap between windows when context is longer than MAX_LENGTH
BATCH_SIZE   = 16    # per-device batch size; reduce to 8 if you hit OOM

# Training arguments — sensible defaults for a single-GPU Colab run
LEARNING_RATE      = 2e-5
NUM_EPOCHS         = 3 if not DEBUG_MODE else 1
WARMUP_RATIO       = 0.1
WEIGHT_DECAY       = 0.01

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model : {MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  MaxLen: {MAX_LENGTH}  |  Stride: {DOC_STRIDE}")

## 3. Load the BioASQ dataset

In [ ]:
# ---------------------------------------------------------------------------
# We use the bigbio/bioasq HuggingFace dataset as the primary source because
# it requires no registration and is already split into train/validation sets.
#
# Config used: bioasq10b_bigbio_qa
#   — Task B, BioASQ 10, BigBio QA schema
#   — Contains factoid, list, yesno, and summary question types
#   — We filter to factoid only.
# ---------------------------------------------------------------------------

print("Downloading BioASQ dataset …")
raw_dataset = load_dataset(
    "bigbio/bioasq",
    name="bioasq10b_bigbio_qa",
    trust_remote_code=True,
)
print(raw_dataset)
print("\nSample keys:", list(raw_dataset["train"][0].keys()))

## 4. Explore & filter to factoid questions

In [ ]:
# Inspect one raw example to understand the schema
sample = raw_dataset["train"][0]
print(json.dumps(
    {k: (v[:200] if isinstance(v, str) else v) for k, v in sample.items()},
    indent=2,
))

In [ ]:
# ---------------------------------------------------------------------------
# The bigbio QA schema fields we care about:
#   question_id  — unique ID
#   question     — the biomedical question
#   context      — concatenated PubMed abstract snippets
#   type         — factoid | list | yesno | summary
#   answers      — list of answer dicts with 'text' key
# ---------------------------------------------------------------------------

def keep_factoid(example):
    return example["type"] == "factoid"

factoid_dataset = raw_dataset.filter(keep_factoid, desc="Filtering factoid questions")

for split, ds in factoid_dataset.items():
    print(f"{split}: {len(ds)} factoid examples")

In [ ]:
# Optional: truncate to DEBUG_N examples per split when in debug mode
if DEBUG_MODE and DEBUG_N is not None:
    factoid_dataset = DatasetDict({
        split: ds.select(range(min(DEBUG_N, len(ds))))
        for split, ds in factoid_dataset.items()
    })
    print("Debug subset sizes:", {k: len(v) for k, v in factoid_dataset.items()})

## 5. Preprocess — locate answer spans in context

In [ ]:
# ---------------------------------------------------------------------------
# BioASQ answers are given as free-text strings but NOT as character offsets.
# We need to find the first occurrence of each answer string inside the context
# so that the QA model can be trained with (start, end) positions.
#
# Strategy:
#   1. Take the first answer text (factoid has one canonical answer).
#   2. Search for it case-insensitively in the context.
#   3. Record char-level start/end.
#   4. Drop examples where the answer isn't found verbatim.
# ---------------------------------------------------------------------------

def add_answer_positions(example):
    """
    Adds 'answer_start' and 'answer_text' fields by locating the
    first answer string within the context text.
    Returns answer_start = -1 if not found (will be dropped later).
    """
    context = example["context"]
    answers = example["answers"]          # list of dicts with 'text' key

    # Take the first listed answer
    answer_text = answers[0]["text"] if answers else ""

    # Case-insensitive substring search
    ctx_lower = context.lower()
    ans_lower = answer_text.lower()
    start_char = ctx_lower.find(ans_lower)

    if start_char != -1:
        # Use the original-cased slice for the exact answer text stored
        actual_text = context[start_char: start_char + len(answer_text)]
    else:
        actual_text = answer_text

    example["answer_text"]  = actual_text
    example["answer_start"] = start_char
    return example


factoid_dataset = factoid_dataset.map(
    add_answer_positions,
    desc="Locating answer spans",
)

# Drop examples where the answer string wasn't found in the context
before = {k: len(v) for k, v in factoid_dataset.items()}
factoid_dataset = factoid_dataset.filter(
    lambda ex: ex["answer_start"] != -1,
    desc="Dropping unanswerable examples",
)
after = {k: len(v) for k, v in factoid_dataset.items()}

for split in before:
    dropped = before[split] - after.get(split, 0)
    print(f"{split}: {after.get(split, 0)} kept, {dropped} dropped")

In [ ]:
# Inspect a preprocessed example
ex = factoid_dataset["train"][0]
print("Question    :", ex["question"])
print("Answer text :", ex["answer_text"])
print("Answer start:", ex["answer_start"])
print("Context [:300]:", ex["context"][:300])

## 6. Tokenize with stride handling

In [ ]:
print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", tokenizer.__class__.__name__)

In [ ]:
# ---------------------------------------------------------------------------
# Tokenization for extractive QA
# ---------------------------------------------------------------------------
# When the context is longer than MAX_LENGTH - (question tokens) - 3 special
# tokens, HuggingFace splits it into overlapping windows ("features").
#
# return_overflowing_tokens=True  → multiple features per example
# return_offsets_mapping=True     → char-to-token offset mapping
# stride=DOC_STRIDE               → overlap between consecutive windows
#
# For each feature we must find which token indices correspond to the answer
# character span.  If the answer is not inside this window we mark it with
# start_positions = end_positions = cls_token_index (unanswerable convention).
# ---------------------------------------------------------------------------

def tokenize_and_align(examples, is_train=True):
    """
    Tokenize a batch of QA examples and compute token-level start/end positions.
    Works for both training (labels needed) and validation (for post-processing).
    """
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",   # only truncate the context, never the question
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # overflow_to_sample_mapping maps each feature index → original example index
    sample_mapping   = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping   = tokenized.pop("offset_mapping")

    start_positions  = []
    end_positions    = []
    example_ids      = []     # for validation post-processing

    for feature_idx, sample_idx in enumerate(sample_mapping):
        # The CLS token is always at position 0 and is used for "no answer" label
        cls_index   = tokenized["input_ids"][feature_idx].index(
            tokenizer.cls_token_id
        )

        # sequence_ids: 0 = question tokens, 1 = context tokens, None = specials
        seq_ids     = tokenized.sequence_ids(feature_idx)
        offsets     = offset_mapping[feature_idx]

        # Record which example this feature belongs to (used at eval time)
        example_ids.append(examples["question_id"][sample_idx])

        if is_train:
            ans_start = examples["answer_start"][sample_idx]
            ans_text  = examples["answer_text"][sample_idx]
            ans_end   = ans_start + len(ans_text) - 1   # inclusive char index

            # Find the context token window boundaries
            ctx_start_tok = next(
                (i for i, s in enumerate(seq_ids) if s == 1), None
            )
            ctx_end_tok   = (
                len(seq_ids)
                - next(i for i, s in enumerate(reversed(seq_ids)) if s == 1)
                - 1
            )

            # If the answer character span falls outside this window, mark as CLS
            window_start_char = offsets[ctx_start_tok][0] if ctx_start_tok else 0
            window_end_char   = offsets[ctx_end_tok][1]   if ctx_end_tok   else 0

            if (
                ctx_start_tok is None
                or ans_start < window_start_char
                or ans_end   > window_end_char
            ):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                # Walk forward to find the token that covers ans_start
                tok_start = ctx_start_tok
                while tok_start <= ctx_end_tok and offsets[tok_start][0] <= ans_start:
                    tok_start += 1
                start_positions.append(tok_start - 1)

                # Walk backward to find the token that covers ans_end
                tok_end = ctx_end_tok
                while tok_end >= ctx_start_tok and offsets[tok_end][1] >= ans_end:
                    tok_end -= 1
                end_positions.append(tok_end + 1)
        else:
            # Validation: store offsets for post-processing (answer extraction)
            tokenized["offset_mapping"] = (
                tokenized.get("offset_mapping", []) + [offsets]
            )

    if is_train:
        tokenized["start_positions"] = start_positions
        tokenized["end_positions"]   = end_positions

    tokenized["example_id"] = example_ids
    return tokenized


# ---------------------------------------------------------------------------
# Build train and validation feature datasets
# ---------------------------------------------------------------------------
print("Tokenizing training split …")
train_features = factoid_dataset["train"].map(
    lambda ex: tokenize_and_align(ex, is_train=True),
    batched=True,
    remove_columns=factoid_dataset["train"].column_names,
    desc="Tokenising train",
)

# Choose the validation split name (some BioASQ configs use 'validation', others 'test')
val_split = "validation" if "validation" in factoid_dataset else "test"
print(f"Tokenizing {val_split} split …")
val_features = factoid_dataset[val_split].map(
    lambda ex: tokenize_and_align(ex, is_train=False),
    batched=True,
    remove_columns=factoid_dataset[val_split].column_names,
    desc="Tokenising validation",
)

print(f"Train features : {len(train_features)}")
print(f"Val features   : {len(val_features)}")
print("Feature columns:", train_features.column_names)

In [ ]:
# Trainer expects torch tensors, not python lists; set the format accordingly.
# Keep example_id as a plain python column (not a tensor) for post-processing.
TRAINER_COLS = ["input_ids", "attention_mask", "token_type_ids",
                "start_positions", "end_positions"]

# token_type_ids may not be present for some tokenisers
trainer_train_cols = [c for c in TRAINER_COLS if c in train_features.column_names]
train_features.set_format(type="torch", columns=trainer_train_cols)

VAL_TENSOR_COLS = ["input_ids", "attention_mask", "token_type_ids"]
trainer_val_cols = [c for c in VAL_TENSOR_COLS if c in val_features.column_names]
val_features.set_format(type="torch", columns=trainer_val_cols, output_all_columns=True)

print("Format set.")

## 7. Load BioBERT for Question Answering

In [ ]:
print("Loading BioBERT …")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Evaluation metrics — Exact Match & token-level F1

In [ ]:
# ---------------------------------------------------------------------------
# Standard SQuAD-style evaluation helpers.
# These operate on plain text strings, not token ids.
# ---------------------------------------------------------------------------

import string, re

def normalize_answer(s: str) -> str:
    """Lower-case, strip punctuation and articles, collapse whitespace."""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = " ".join(s.split())
    return s


def exact_match_score(prediction: str, ground_truth: str) -> float:
    return float(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score(prediction: str, ground_truth: str) -> float:
    pred_tokens  = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    common = collections.Counter(pred_tokens) & collections.Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)


print("Metric helpers defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Post-processing: convert logit outputs → answer strings
# ---------------------------------------------------------------------------
# The Trainer gives us (start_logits, end_logits) for every feature (window).
# For each original example we pick the best non-null span across all windows.
# ---------------------------------------------------------------------------

N_BEST   = 20     # candidate (start, end) pairs to consider per feature
MAX_SPAN = 30     # maximum answer length in tokens

def postprocess_qa_predictions(
    examples,           # original (un-tokenised) examples
    features,           # tokenised features (with offset_mapping)
    raw_predictions,    # (start_logits, end_logits) arrays, shape [num_features, seq_len]
):
    """
    For each example, select the best answer span across all its windows.
    Returns a dict {example_id: predicted_answer_string}.
    """
    start_logits, end_logits = raw_predictions

    # Map example_id → list of feature indices
    example_to_features = collections.defaultdict(list)
    for i, ex_id in enumerate(features["example_id"]):
        example_to_features[ex_id].append(i)

    predictions = {}

    for ex in examples:
        ex_id   = ex["question_id"]
        context = ex["context"]
        best_answer = {"text": "", "score": float("-inf")}

        for feat_idx in example_to_features[ex_id]:
            start_log = start_logits[feat_idx]
            end_log   = end_logits[feat_idx]
            offsets   = features["offset_mapping"][feat_idx]
            seq_ids   = features[feat_idx].get("sequence_ids", None)  # may not exist

            # Determine which tokens belong to the context (sequence id == 1)
            # If sequence_ids isn't stored, fall back to non-zero offsets after CLS.
            if seq_ids is not None:
                ctx_mask = [1 if s == 1 else 0 for s in seq_ids]
            else:
                # Rough heuristic: skip question tokens by finding the first
                # (0, 0) offset after the first real token (SEP separates Q & C).
                sep_seen  = 0
                ctx_mask  = []
                for off in offsets:
                    if off == (0, 0):
                        sep_seen += 1
                        ctx_mask.append(0)
                    else:
                        ctx_mask.append(1 if sep_seen >= 1 else 0)

            # Zero out logits for non-context tokens
            start_log = [s if ctx_mask[i] else float("-inf") for i, s in enumerate(start_log)]
            end_log   = [s if ctx_mask[i] else float("-inf") for i, s in enumerate(end_log)]

            # Get top-N start and end positions
            top_starts = sorted(range(len(start_log)), key=lambda i: start_log[i], reverse=True)[:N_BEST]
            top_ends   = sorted(range(len(end_log)),   key=lambda i: end_log[i],   reverse=True)[:N_BEST]

            for start in top_starts:
                for end in top_ends:
                    if (
                        end < start
                        or (end - start + 1) > MAX_SPAN
                        or offsets[start] == (0, 0)
                        or offsets[end]   == (0, 0)
                    ):
                        continue
                    score = start_log[start] + end_log[end]
                    if score > best_answer["score"]:
                        char_start = offsets[start][0]
                        char_end   = offsets[end][1]
                        best_answer = {
                            "text":  context[char_start:char_end],
                            "score": score,
                        }

        predictions[ex_id] = best_answer["text"] if best_answer["text"] else ""

    return predictions


print("Post-processing function defined.")

In [ ]:
# ---------------------------------------------------------------------------
# compute_metrics is called by Trainer after each evaluation epoch.
# eval_pred contains (predictions, label_ids); for QA, predictions is a tuple
# (start_logits, end_logits) and label_ids is (start_positions, end_positions).
# ---------------------------------------------------------------------------

# We capture val_features and factoid_dataset via closure.
val_examples = factoid_dataset[val_split]

def compute_metrics(eval_pred):
    start_logits, end_logits = eval_pred.predictions

    preds = postprocess_qa_predictions(
        examples=val_examples,
        features=val_features,
        raw_predictions=(start_logits, end_logits),
    )

    em_scores, f1_scores = [], []
    for ex in val_examples:
        ex_id   = ex["question_id"]
        gold    = ex["answer_text"]
        pred    = preds.get(ex_id, "")
        em_scores.append(exact_match_score(pred, gold))
        f1_scores.append(f1_score(pred, gold))

    return {
        "exact_match": round(100.0 * np.mean(em_scores), 2),
        "f1":          round(100.0 * np.mean(f1_scores), 2),
    }


print("compute_metrics defined.")

## 9. Fine-tuning with HuggingFace Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),      # mixed precision only on CUDA
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",                    # disable wandb / MLflow by default
    dataloader_num_workers=0,            # 0 avoids multiprocessing issues on Colab
)

print("TrainingArguments ready.")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=val_features,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialised. Starting training …")
train_result = trainer.train()

print("\nTraining complete.")
print(f"  Train loss     : {train_result.training_loss:.4f}")
print(f"  Train runtime  : {train_result.metrics['train_runtime']:.1f}s")

## 10. Evaluate on validation set

In [ ]:
eval_metrics = trainer.evaluate()

print("\n===== Evaluation Results =====")
for k, v in eval_metrics.items():
    print(f"  {k}: {v}")

## 11. Save the fine-tuned model

In [ ]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

## 12. Inference function

In [ ]:
from transformers import pipeline as hf_pipeline

qa_pipe = hf_pipeline(
    "question-answering",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if DEVICE == "cuda" else -1,   # -1 → CPU/MPS handled automatically
    handle_impossible_answer=False,
)


def predict_answer(question: str, context: str) -> dict:
    """
    Given a biomedical question and a PubMed abstract, return the
    extracted answer span and confidence score.

    Args:
        question : biomedical factoid question
        context  : PubMed abstract text

    Returns:
        dict with keys 'answer' (str) and 'score' (float)
    """
    result = qa_pipe(question=question, context=context)
    return {"answer": result["answer"], "score": round(result["score"], 4)}


print("Inference function ready.")

## 13. Demo predictions

In [ ]:
# Run inference on the first 3 validation examples
print("Sample predictions on validation examples:\n")
for i in range(min(3, len(val_examples))):
    ex = val_examples[i]
    result = predict_answer(ex["question"], ex["context"])
    print(f"[{i+1}] Question  : {ex['question']}")
    print(f"    Gold answer: {ex['answer_text']}")
    print(f"    Prediction : {result['answer']}  (score: {result['score']})")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Custom inference — paste your own question and abstract here
# ---------------------------------------------------------------------------
MY_QUESTION = "What is the primary mechanism of action of metformin?"

MY_ABSTRACT = (
    "Metformin is a biguanide drug used as a first-line treatment for type 2 "
    "diabetes mellitus. Its primary mechanism of action involves the inhibition "
    "of hepatic glucose production (gluconeogenesis) via activation of AMP-activated "
    "protein kinase (AMPK). Additionally, metformin improves peripheral insulin "
    "sensitivity and reduces intestinal glucose absorption."
)

output = predict_answer(MY_QUESTION, MY_ABSTRACT)
print(f"Question : {MY_QUESTION}")
print(f"Answer   : {output['answer']}")
print(f"Score    : {output['score']}")